In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import EarthLocation, get_sun, get_body  # <-- Uses get_body for modern Astropy
from astroplan import Observer, FixedTarget
from astroplan.moon import moon_illumination
from geopy.geocoders import Nominatim
import datetime
import zoneinfo
import warnings
import urllib.request
import json
from astroplan import TargetAlwaysUpWarning

# Import Widget libraries
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", category=TargetAlwaysUpWarning)

# 1. Initialize Geolocator
geolocator = Nominatim(user_agent="astro_scheduler_modern_widget_2026")

# 2. Setup the Interactive Control Elements
location_type = widgets.ToggleButtons(
    options=['Address String', 'Lat / Lon Coordinates'],
    description='Loc Mode:',
    button_style='primary'
)

address_input = widgets.Text(
    value="Wiesenstrasse 58 64331 Weiterstadt Germany",
    description="Address:",
    layout=widgets.Layout(width="400px")
)

lat_input = widgets.BoundedFloatText(
    value=49.9, min=-90.0, max=90.0, step=0.01,
    description="Latitude:", layout=widgets.Layout(width="180px")
)

lon_input = widgets.BoundedFloatText(
    value=8.6, min=-180.0, max=180.0, step=0.01,
    description="Longitude:", layout=widgets.Layout(width="180px")
)

here_button = widgets.Button(
    description="Here", button_style="info", icon="map-marker",
    layout=widgets.Layout(width="80px")
)

date_input = widgets.DatePicker(
    value=datetime.date(2026, 6, 27),
    description="Obs Date:"
)

today_button = widgets.Button(
    description="Today", button_style="info", icon="calendar",
    layout=widgets.Layout(width="80px")
)

threshold_input = widgets.IntSlider(
    value=-10, min=-18, max=0, step=1,
    description="Sun Alt Max:",
)

targets_input = widgets.Textarea(
    value="NGC6960, NGC6992, NGC7635, M31, M42, M51, M13, M16",
    description="DSO Targets:",
    placeholder="Enter targets separated by commas (e.g. M31, M42)",
    layout=widgets.Layout(width="490px", height="60px")
)

run_button = widgets.Button(
    description="Generate Plot", button_style="success", icon="check",
    layout=widgets.Layout(width="150px")
)

output_area = widgets.Output()

# Containers for switching inputs dynamically
loc_address_box = widgets.HBox([address_input, here_button])
loc_coords_box = widgets.HBox([lat_input, lon_input, here_button])
dynamic_loc_container = widgets.VBox([loc_address_box])

def on_loc_type_change(change):
    if change['new'] == 'Address String':
        dynamic_loc_container.children = (loc_address_box,)
    else:
        dynamic_loc_container.children = (loc_coords_box,)

location_type.observe(on_loc_type_change, names='value')

# Quick actions
def set_today_date(b=None):
    date_input.value = datetime.date.today()

def set_current_location(b=None):
    address_input.value = "Locating..."
    try:
        with urllib.request.urlopen("https://ipapi.co/json/", timeout=5) as response:
            data = json.loads(response.read().decode())
            lat = data.get("latitude")
            lon = data.get("longitude")
            
            if lat and lon:
                lat_input.value = float(lat)
                lon_input.value = float(lon)
                rev_loc = geolocator.reverse(f"{lat}, {lon}", timeout=5)
                address_input.value = rev_loc.address if rev_loc else f"{lat}, {lon}"
            else:
                address_input.value = "Error capturing IP"
    except Exception:
        address_input.value = "Auto-locate failed"

today_button.on_click(set_today_date)
here_button.on_click(set_current_location)

# 3. Core calculation and tracking system
def generate_schedule_plot(b=None):
    with output_area:
        clear_output(wait=True)
        
        try:
            raw_targets = [t.strip() for t in targets_input.value.split(",") if t.strip()]
            if not raw_targets:
                print("Error: Target catalog list is completely empty.")
                return
            
            if location_type.value == 'Address String':
                loc_str = address_input.value
                if loc_str in ["Locating...", "Auto-locate failed", "Error capturing IP"]:
                    print("Please insert or resolve a location address layout.")
                    return
                location = geolocator.geocode(loc_str)
                if not location:
                    print(f"Error: Could not resolve address: '{loc_str}'")
                    return
                lat_val, lon_val, alt_val = location.latitude, location.longitude, location.altitude
                display_loc = loc_str.split(',')[0]
            else:
                lat_val, lon_val, alt_val = lat_input.value, lon_input.value, 0.0
                display_loc = f"Lat: {lat_val:.2f}, Lon: {lon_val:.2f}"

            earth_location = EarthLocation(
                lat=lat_val * u.deg, lon=lon_val * u.deg, 
                height=alt_val * u.m if alt_val else 0 * u.m
            )
            observer = Observer(location=earth_location, name="Target Obs Site")

            date_str = date_input.value.strftime("%Y-%m-%d")
            base_date = Time(date_str) 
            scan_start = base_date + 12 * u.hour 
            time_grid = scan_start + np.linspace(0, 18, 500) * u.hour

            sun_altaz = observer.altaz(time_grid, get_sun(time_grid))
            dark_mask = sun_altaz.alt.deg <= float(threshold_input.value)
            dark_times = time_grid[dark_mask]

            if len(dark_times) == 0:
                print(f"No darkness window caught (Sun <= {threshold_input.value}°) for this date/location.")
                return

            local_tz = datetime.datetime.now().astimezone().tzinfo
            local_datetimes = [dt.replace(tzinfo=datetime.timezone.utc).astimezone(local_tz) for dt in dark_times.datetime.ravel()]
            window_start, window_end = local_datetimes[0], local_datetimes[-1]

            # Calculate Moon Phase / Illumination details
            moon_illums = moon_illumination(dark_times)
            avg_moon_illum = np.mean(moon_illums) * 100.0
            moon_label = f"MOON ({avg_moon_illum:.1f}%)"
            
 #   moon_phase_percent = moon_illumination(midpoint_time) * 100
#    moon_label = f"MOON ({moon_phase_percent:.1f}% Illum)"

            # Calculate Moon position parameters across the time grid using get_body
            moon_coords = get_body('moon', dark_times, location=earth_location)
            moon_altaz = observer.altaz(dark_times, moon_coords)
            moon_alts = moon_altaz.alt.deg
            moon_azs = moon_altaz.az.deg

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 11), sharex=True)
            plotted_count = 0

            # Plot dynamic targets
            for t_name in raw_targets:
                try:
                    target = FixedTarget.from_name(t_name)
                    altaz_positions = observer.altaz(dark_times, target)
                    alts, azs = altaz_positions.alt.deg, altaz_positions.az.deg
                    
                    p, = ax1.plot(local_datetimes, alts, label=t_name, lw=3)
                    color = p.get_color()
                    ax2.plot(local_datetimes, azs, label=t_name, lw=3)
                    plotted_count += 1
                    
                    transit_time = observer.target_meridian_transit_time(dark_times[0], target, which='next')
                    transit_local = transit_time.datetime.replace(tzinfo=datetime.timezone.utc).astimezone(local_tz)
                    
                    if window_start <= transit_local <= window_end:
                        transit_alt = observer.altaz(transit_time, target).alt.deg
                        if transit_alt > 0:
                            ax1.plot(transit_local, transit_alt, marker='o', color=color, markersize=10)
                            ax1.text(transit_local, transit_alt + 2, f"{transit_local.strftime('%H:%M')}", 
                                     fontsize=12, ha='center', weight='bold', color=color)
                            ax2.axvline(transit_local, color=color, linestyle=':', alpha=0.5, lw=1.5)
                except Exception:
                    pass

            # Plot Moon trajectory lines (Thick dashed black line to differentiate)
            ax1.plot(local_datetimes, moon_alts, label=moon_label, color="black", linestyle="--", lw=4, alpha=0.8)
            ax2.plot(local_datetimes, moon_azs, label=moon_label, color="black", linestyle="--", lw=4, alpha=0.8)

            ax1.axvspan(window_start, window_end, color='midnightblue', alpha=0.10)
            ax2.axvspan(window_start, window_end, color='midnightblue', alpha=0.10)

            ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=local_tz))
            ax2.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=local_tz))

            ax1.set_title(f"Target Altitude (Sun <= {threshold_input.value}° Window) | Moon Illum: {avg_moon_illum:.1f}%", fontsize=16, pad=10, weight='bold')
            ax1.set_ylabel("Altitude (Degrees)", fontsize=14, weight='bold')
            ax1.set_ylim(0, 90)
            ax1.axhline(20, color='red', lw=2, linestyle='--', alpha=0.8, label='Min Imaging Alt (20°)')
            ax1.tick_params(axis='both', labelsize=12)
            ax1.grid(True, linestyle="--", alpha=0.4)

            ax2.set_title("Target & Moon Azimuth Schedule", fontsize=16, pad=10, weight='bold')
            ax2.set_xlabel(f"Local Clock Time ({window_start.strftime('%Z')} / HH:MM)", fontsize=14, weight='bold')
            ax2.set_ylabel("Azimuth (Degrees)", fontsize=14, weight='bold')
            ax2.set_yticks([0, 90, 180, 270, 360])
            ax2.set_yticklabels(['0° (N)', '90° (E)', '180° (S)', '270° (W)', '360° (N)'])
            ax2.set_ylim(0, 360)
            ax2.tick_params(axis='both', labelsize=12)
            ax2.grid(True, linestyle="--", alpha=0.4)

            plt.suptitle(f"Astrophotography Tracking Schedule\nNight of {date_str} | Location: {display_loc}", fontsize=18, y=0.98, weight='bold')

            handles, labels = ax1.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax1.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True, fontsize=12)

            plt.tight_layout()
            plt.show()
            
        except Exception as main_err:
            print(f"An unexpected tracking error occurred: {main_err}")

run_button.on_click(generate_schedule_plot)

# Compile layout architecture
controls = widgets.VBox([
    location_type,
    dynamic_loc_container,
    widgets.HBox([date_input, today_button]),
    threshold_input,
    targets_input,
    run_button
])

display(controls, output_area)
generate_schedule_plot()

Output()